In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from scripts.set_seed import set_seed

from src.model_factory import build_model

from src.dataset import MSADataset, CLASS_MAP

from src.embedder import MSAEmbedder

from src.explainibilty import (
    make_zero_baseline,
    explain_predictions,
    print_results,
    visualize_sequence_explanations,
    visualize_attention_explanations,
    compute_attention_weights,
    compute_saliency
)

In [ ]:
# ============================================================
# LOAD CONFIG, SEED, AND SAVED MODEL
# ============================================================

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

RUN_DIR = Path(run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["experiment"]["seed"])

model = build_model(cfg)
model.load_state_dict(torch.load(RUN_DIR / "final_model.pt"))
print(f"Loaded model from {RUN_DIR / 'final_model.pt'}")

print(json.dumps(cfg, indent=2))

In [ ]:
# ============================================================
# LOAD AND EMBED TEST DATA
# ============================================================

# Inference on a fixed set of sequences
test_data1 = MSADataset(['/content/drive/MyDrive/Thesis data/MSAs/ex_claudin_msa.fasta'],
    [-1], test_data=True,
)
test_data1_seq_len = test_data1.getSequenceLength()
test_data1_seq_ids, test_data1_seqs = test_data1.getSequences()

print(f'# Test sequences: {len(test_data1_seqs)}')

# Embed either in MSA mode or independently depending on the config
embedder = MSAEmbedder()
if cfg['data']['use_msa_mode']:
    test_data1_embeddings = embedder.embed_msa(sequences=test_data1_seqs, seq_length=test_data1_seq_len, max_msa_depth=len(test_data1_seqs))
else:
    test_data1_embeddings = embedder.embed_sequences_per_residue(sequences=test_data1_seqs, seq_length=test_data1_seq_len, batch_size=1)

print(f'Embeddings shape: {test_data1_embeddings.shape}')

# Run inference with final model
model.eval()
with torch.no_grad():
    logits = model(test_data1_embeddings.to(device))
    probs = torch.softmax(logits, dim=1)
    pred  = probs.argmax(dim=1)

for i, cls in enumerate(pred.cpu().numpy()):
    print(f"\n({i}) {test_data1_seq_ids[i]}: {test_data1_seqs[i]}")
    print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs[i, cls]:.3f}")

# Save preds
preds_df = pd.DataFrame({
    "seq_id": test_data1_seq_ids,
    "sequence": test_data1_seqs,
    "predicted_class": [CLASS_MAP[cls] for cls in pred.cpu().numpy()],
    "confidence": probs.cpu().numpy().tolist(),  # confidence across all classes
})
preds_df.to_csv(RUN_DIR / "inference/predictions/test_predictions1.csv", index=False)

In [ ]:
# ============================================================
# EXPLAINIBILTY - IG
# ============================================================

test_data1_seq_ids_to_explain = [test_data1_seq_ids[i] for i in [6, 15, 17]]
test_data1_seqs_to_explain = [test_data1_seqs[i] for i in [6, 15, 17]]
test_data1_embeddings_to_explain = test_data1_embeddings[[6, 15, 17], :, :]

baseline_embedding = make_zero_baseline(test_data1_embeddings_to_explain.shape[1], embed_dim=768)

model.eval()
with torch.no_grad():
    logits = model(test_data1_embeddings_to_explain.to(device))
    probs = torch.softmax(logits, dim=1)
    predicted_classes = probs.argmax(1)
    confidences = probs.max(1)[0]

true_classes = predicted_classes  # Assume model is correct for IG

# ── Compute IG explanations ──
results1 = explain_predictions(
    model,
    test_data1_seq_ids_to_explain,
    test_data1_seqs_to_explain,
    test_data1_embeddings_to_explain,
    baseline_embedding,
    predicted_classes,
    confidences,
    true_classes,
    k=10, n_steps=100, device=device, run_ablation=True,
)
print_results(results1)

# Save explanations
with open(RUN_DIR / "inference/explanations/explanations1_ig.json", "w") as f:
    json.dump(results1, f, indent=2)

In [ ]:
visualize_sequence_explanations(results=results1, class_names=list(CLASS_MAP.values()), max_sequences=5, save_name=RUN_DIR / "inference/explanations/explanations1_ig_viz")

In [ ]:
# ============================================================
# EXPLAINIBILTY - ATTENTION / SALIENCY
# ============================================================

if cfg['model']['use_attention']:
    attention_weights = compute_attention_weights(model, test_data1_embeddings_to_explain.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations1_attn"
else:
    attention_weights = compute_saliency(model, test_data1_embeddings_to_explain.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations1_saliency"
    
visualize_attention_explanations(attention_weights, class_names=list(CLASS_MAP.values()), save_name=save_name)

In [ ]:
# ============================================================
# LOAD AND EMBED TEST DATA SPECIFIED IN CONFIG
# ============================================================

# Inference on a fixed set of sequences
test_data2 = MSADataset([f'/content/drive/MyDrive/Thesis data/MSAs/{cfg["evaluation"]["test_data"]}'],
    [-1], test_data=True,
)
test_data2_seq_len = test_data2.getSequenceLength()
test_data2_seq_ids, test_data2_seqs = test_data2.getSequences()

print(f'# Test sequences: {len(test_data2_seqs)}')

# Embed either in MSA mode or independently depending on the config
embedder = MSAEmbedder()
if cfg['data']['use_msa_mode']:
    test_data2_embeddings = embedder.embed_msa(sequences=test_data2_seqs, seq_length=test_data2_seq_len, max_msa_depth=len(test_data2_seqs))
else:
    test_data2_embeddings = embedder.embed_sequences_per_residue(sequences=test_data2_seqs, seq_length=test_data2_seq_len, batch_size=1)

print(f'Embeddings shape: {test_data2_embeddings.shape}')

# Run inference with final model
model.eval()
with torch.no_grad():
    logits = model(test_data2_embeddings.to(device))
    probs = torch.softmax(logits, dim=1)
    pred  = probs.argmax(dim=1)

for i, cls in enumerate(pred.cpu().numpy()):
    print(f"\n({i}) {test_data2_seq_ids[i]}: {test_data2_seqs[i]}")
    print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs[i, cls]:.3f}")

# Save preds
preds_df = pd.DataFrame({
    "seq_id": test_data2_seq_ids,
    "sequence": test_data2_seqs,
    "predicted_class": [CLASS_MAP[cls] for cls in pred.cpu().numpy()],
    "confidence": probs.cpu().numpy().tolist(),  # confidence across all classes
})
preds_df.to_csv(RUN_DIR / "inference/predictions/test_predictions2.csv", index=False)

In [ ]:
# ============================================================
# EXPLAINIBILTY - IG
# ============================================================

baseline_embedding = make_zero_baseline(test_data2_embeddings_to_explain.shape[1], embed_dim=768)

model.eval()
with torch.no_grad():
    logits = model(test_data2_embeddings_to_explain.to(device))
    probs = torch.softmax(logits, dim=1)
    predicted_classes = probs.argmax(1)
    confidences = probs.max(1)[0]

true_classes = predicted_classes  # Assume model is correct for IG

# ── Compute IG explanations ──
results2 = explain_predictions(
    model,
    test_data2_seq_ids_to_explain,
    test_data2_seqs_to_explain,
    test_data2_embeddings_to_explain,
    baseline_embedding,
    predicted_classes,
    confidences,
    true_classes,
    k=10, n_steps=100, device=device, run_ablation=True,
)
print_results(results2)

# Save explanations
with open(RUN_DIR / "inference/explanations/explanations2_ig.json", "w") as f:
    json.dump(results2, f, indent=2)

In [ ]:
visualize_sequence_explanations(results=results2, class_names=list(CLASS_MAP.values()), max_sequences=5, save_name=RUN_DIR / "inference/explanations/explanations2_ig_viz")

In [ ]:
# ============================================================
# EXPLAINIBILTY - ATTENTION / SALIENCY
# ============================================================

if cfg['model']['use_attention']:
    attention_weights = compute_attention_weights(model, test_data2_embeddings_to_explain.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations2_attn"
else:
    attention_weights = compute_saliency(model, test_data2_embeddings_to_explain.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations2_saliency"

visualize_attention_explanations(attention_weights, class_names=list(CLASS_MAP.values()), save_name=save_name)